In [0]:
# DLT works with 3 types of datasets
# 1. Streaming Table (Permanent/Temporary) - Used as Append Data Sources, Incremental Data
# 2. Materialized Views - Used for transformations, aggregations or computations 
# 3. Views - Used for intermediate transformations, not stored in Target Schema
# DLT pipelines are powered by delta lake, so we get all the power of Delta Lake as well

import dlt

In [0]:
_order_status = spark.conf.get("custom.orderStatus","NA")

In [0]:
# Create a streaming table for Orders
# Delta table has property to work as both batch and stream mode.  
# by default name of function will be name of streaming table : orders_bronze 
@dlt.table(
    table_properties = {"quality" : "bronze"},
    comment = "Order bronze table",
    name = "orders_bronze_refactored"
)
def get_orders_bronze():
    df = spark.readStream.table("dbcertification.easewithdata.orders_raw_refactored")
    return df

In [0]:
# Create a streaming table for Orders Autoloader

@dlt.table(
    table_properties = {"quality" : "bronze"},
    comment = "Order autoloader",
    name = "orders_autoloader_bronze_refactored"
)
def func():
    df = (
        spark
        .readStream
        .format("cloudFiles")
        .option("cloudFiles.schemaHints","o_orderkey long, o_custkey long, o_orderstatus string, o_totalprice decimal(18,2), o_orderdate date, o_orderpriority string, o_clerk string, o_shippriority int, o_comment string")
        .option("cloudFiles.schemaLocation","/Volumes/dbcertification/easewithdata/volume_easewithdata/dlt_autoloader_landing_refactored/autoloader/schemas/1")
        .option("cloudFiles.format","CSV")
        .option("pathGlobFilter","*.csv")
        .option("cloudFiles.schemaEvolutionMode","none")
        .load("/Volumes/dbcertification/easewithdata/volume_easewithdata/dlt_autoloader_landing_refactored/files/")
    )
    return df

In [0]:
dlt.create_streaming_table("orders_union_bronze_refactored") #empty table

# Now we have to append data from both streaming tables one by one into this above table

# Append Flow
@dlt.append_flow(
    target = "orders_union_bronze_refactored"  
 )
def order_delta_append():
    df = dlt.read_stream("Live.orders_bronze_refactored")
    return df

# Append Flow
@dlt.append_flow(
     target = "orders_union_bronze_refactored"  
 )
def order_autoloader_append():
     df = dlt.read_stream("Live.orders_autoloader_bronze_refactored")
     return df

# use of append flow is appending multiple streaming tables into one, incrementally

In [0]:
# Create a Materialized View for Customers
# dlt.table will only be used for streaming table and materialized view 

@dlt.table(
    table_properties = {"quality" : "bronze"},
    comment = "Customer bronze table",
    name = "customer_bronze_refactored"
)
def cust_bronze():
    df = spark.read.table("dbcertification.easewithdata.customer_raw_refactored")
    return df

In [0]:
# Create a dlt view to join orders with customers
# LIVE keyword is used to reference DLT tables and views created in same pipeline

@dlt.view(
    comment = "Joined View"
)
def joined_vw():
    df_c = dlt.read("LIVE.customer_bronze_refactored")
    df_o = dlt.read_stream("LIVE.orders_union_bronze_refactored")

    df_join = df_o.join(df_c, how = "left_outer", on = df_c.c_custkey == df_o.o_custkey)
      
    return df_join 


In [0]:
# Create MV to add new column
from pyspark.sql.functions import current_timestamp

@dlt.table(
    table_properties = {"quality" : "silver"},
    comment = "joined table",
    name = "orders_silver_refactored"
)
def get_joined_silver():
    df = dlt.read_stream("LIVE.joined_vw").withColumn("__insert_date",current_timestamp())
    return df

In [0]:
# Aggregate based on c_mktsegment and find the count of order  (c_orderkey)
# add one more column Aggregate based on order_total_price and find the sum of total_price

from pyspark.sql.functions import current_timestamp, count, sum

@dlt.table(
    table_properties = {"quality" : "gold"},
    comment = "Orders Aggregated table",
    name = "orders_agg_gold_refactored"
)
def orders_agg_gold():
    df = dlt.read("LIVE.orders_silver_refactored")

    df_final = df.groupBy("c_mktsegment").agg(count("o_orderkey").alias("count_orders"),sum("o_totalprice").alias("sum_totalprice")).withColumn("__insert_date",current_timestamp())
    
    return df_final

In [0]:
from pyspark.sql.functions import current_timestamp, count, sum
 
for _status in _order_status.split(","):
    @dlt.table(
        table_properties = {"quality" : "gold"},
        comment = "Orders Aggregated table",
        name = f"orders_agg_{_status}_gold_refactored"
        )
    def func():
        df = dlt.read("LIVE.orders_silver_refactored")

        df_final = df.where(f"o_orderStatus = '{_status}'").groupBy("c_mktsegment").agg(count("o_orderkey").alias("count_orders"),sum("o_totalprice").alias("sum_totalprice")).withColumn("__insert_date",current_timestamp())
    
        return df_final